# GoldenCheetah 데이터 탐색

## 분석 목적

GoldenCheetah 공개 데이터가 훈련 분석과 오늘의 훈련 추천 프로젝트에
사용할 수 있는지 확인한다.

## 현재까지 확인한 내용

- 전체 운동 기록: 731개
- 자전거 운동: 592개
- 파워 센서 포함: 469개
- 심박 센서 포함: 465개
- 파워와 심박 모두 포함: 373개
- 라이딩마다 센서와 요약 지표 구성이 다름

In [10]:
import sys
from pathlib import Path

import json

import pandas as pd

project_root = Path("..").resolve()

print("Python 경로:", sys.executable)
print("프로젝트 경로:", project_root)
print("pandas 버전:", pd.__version__)

json_path = (
    project_root
    / "data"
    / "raw"
    / "033874ce-e20d-44ba-9cc9-125030b6662f"
    / "{033874ce-e20d-44ba-9cc9-125030b6662f}.json"
)

Python 경로: /Users/hooni/Documents/ChatGPT/Cycling App/.venv/bin/python
프로젝트 경로: /Users/hooni/Documents/ChatGPT/Cycling App
pandas 버전: 3.0.5


## 1. JSON 데이터 불러오기

JSON 파일을 Python 자료형으로 불러온다.
전체 운동 중 `Bike` 기록만 선택하여 pandas 분석에 사용한다.

In [12]:
with json_path.open("r", encoding="utf-8") as file:
    cycling_data = json.load(file)

rides = cycling_data["RIDES"]

bike_rides = [ride for ride in rides if ride["sport"] == "Bike"]

print("전체 운동 수:", len(rides))
print("자전거 운동 수:", len(bike_rides))

전체 운동 수: 731
자전거 운동 수: 592


라이드 기록의 `data`는 15자리의 대문자 알파벳 문자열을 값으로 가지는데, 해당 운동에 어떤 센서 데이터가 포함되어 있는지 나타낸다.

| 문자 | 데이터 |
|---|---|
| `T` | 시간 |
| `D` | 거리 |
| `S` | 속도 |
| `P` | 파워 |
| `H` | 심박수 |
| `C` | 케이던스 |
| `N` | 토크 |
| `A` | 고도 |
| `G` | GPS |
| `L` | 경사도 |
| `W` | 풍속 |
| `E` | 온도 |
| `V` | 좌우 페달 데이터 |
| `O` | 근육 산소 관련 데이터 |
| `R` | Garmin 러닝 다이내믹스 |

In [18]:
first_ride = bike_rides[0]

print(first_ride.keys())
print("첫 라이드의 날짜", first_ride["date"])
print("첫 라이드 기록의 센서 목록:", first_ride["data"])

dict_keys(['date', 'data', 'sport', 'METRICS'])
첫 라이드의 날짜 2005/06/25 14:26:00 UTC
첫 라이드 기록의 센서 목록: TDS-H--A-L-----


첫 라이드의 경우 `TDS-H--A-L-----`로, 시간, 거리, 속도, 심박수, 고도, 경사도 데이터를 포함하고 있다.

## 2. 자전거 기록을 표로 변환

592개의 자전거 기록은 딕셔너리로 구성되어 있다.
pandas의 `DataFrame`을 이용하여 행과 열로 구성된 표로 변환한다.

In [20]:
bike_raw_df = pd.DataFrame(bike_rides)

print("자료형", type(bike_raw_df))
print("표 크기:", bike_raw_df.shape)
print("열 이름", bike_raw_df.columns)

bike_raw_df.head()

자료형 <class 'pandas.DataFrame'>
표 크기: (592, 5)
열 이름 Index(['date', 'data', 'sport', 'METRICS', 'XDATA'], dtype='str')


,date,data,sport,METRICS,XDATA
0,2005/06/25 14:26:00 UTC,TDS-H--A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN
1,2005/06/27 08:39:00 UTC,TDS-H--A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN
2,2005/06/28 17:44:00 UTC,TDS-HC-A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN
3,2005/07/06 18:02:00 UTC,TDS-H--A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN
4,2005/07/10 09:39:00 UTC,TDS-H--A-L-----,Bike,"{'a_skiba_variability_index': 'nan', 'a_coggam...",NaN


- 표는 592개의 행과 5개의 열로 구성되어 있다.
- 기본 열은 `date`, `data`, `sport`, `METRICS`, `XDATA`이다.
- `METRICS`는 아직 하나의 딕셔너리로 저장되어 있다.
- `XDATA`는 일부 라이딩에만 존재하므로 대부분 결측값으로 표시된다.

## 3. METRICS 펼치기

각 라이딩의 `METRICS`에는 운동 시간, 거리, 파워, 심박수 등
여러 요약 지표가 딕셔너리 형태로 저장되어 있다.

`METRICS`의 각 키를 DataFrame의 개별 열로 변환한다.

In [21]:
metrics_df = pd.json_normalize(bike_raw_df["METRICS"])

print("METRICS 표 크기", metrics_df.shape)
print("앞쪽 열 10개:")
print(metrics_df.columns[:10])

METRICS 표 크기 (592, 228)
앞쪽 열 10개:
Index(['a_skiba_variability_index', 'a_coggam_variability_index', 'ride_count',
       'workout_time', 'time_riding', 'total_distance', 'climb_rating',
       'athlete_weight', 'elevation_gain', 'elevation_loss'],
      dtype='str')


In [22]:
sample_columns = [
    "workout_time",
    'time_riding',
    "total_distance",
    'elevation_gain',
    "average_power",
    "average_hr",
    "coggan_tss",
    "coggan_if",
]

metrics_df[sample_columns].head()

,workout_time,time_riding,total_distance,elevation_gain,average_power,average_hr,coggan_tss,coggan_if
0,4800.00000,4780.00000,35.32750,367.00000,NaN,"[146.91667, 960.00000]",NaN,NaN
1,6476.00000,6325.00000,35.01400,467.50000,NaN,"[124.97267, 6476.00000]",NaN,NaN
2,2156.00000,2156.00000,17.65300,92.00000,NaN,"[157.04592, 2156.00000]",NaN,NaN
3,6360.00000,6320.00000,31.04950,512.00000,NaN,"[120.05660, 1272.00000]",NaN,NaN
4,4680.00000,4660.00000,32.77000,471.00000,NaN,"[146.42735, 936.00000]",NaN,NaN


- 592개 자전거 기록의 `METRICS`를 펼치자 228개의 지표가 나타났다.
- 라이딩마다 포함된 지표가 달라 전체 지표 수가 많아졌다.
- 이전 .py 파일에서 보았듯이, 평균 파워와 평균 심박은 일부 행에서 리스트 형태로 저장되어 있다.
- 파워 센서가 없는 초기 라이딩에서는 파워, TSS, IF가 결측값으로 표시된다.
- 228개 지표를 모두 사용하지 않고 분석 목적에 필요한 지표만 선택해야 한다.